In [1]:
# ============================================================
# FULLY CONNECTED GRAPH NEURAL NETWORK — Classification
# ============================================================
# In a fully connected (complete) graph:
#   - Every particle is a NODE with features (pT, eta, phi, flag)
#   - Every pair of particles is connected by an EDGE
#   - Edge features are computed from the pair (ΔR, Δη, Δφ, mass, kT, etc.)
#   - Message passing: nodes exchange information with ALL other nodes
#
# This is the most general graph structure — no kNN or radius cut.
# The model learns which connections matter through attention/weighting.
#
# Architecture:
#   1. Node encoder: per-particle features → latent node features
#   2. Edge encoder: pairwise features → latent edge features
#   3. Message passing layers (fully connected):
#      - Compute messages on all edges
#      - Aggregate messages to update node features
#      - Optionally update edge features
#   4. Global pooling → fixed-size representation
#   5. Combine with event-level features → classify

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""           # Force CPU

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)

# ============================================================
# 1. LOAD AND PARSE DATA
# ============================================================

sig_df = pd.read_csv('CoLBTHydro.csv', header=None)
bkg_df = pd.read_csv('Jewel.csv', header=None)

MAX_PARTICLES = 50
N_FEAT_PER_PARTICLE = 4        # pT, eta, phi, flag
OBJ_START = 8


def parse_data(df, max_particles=MAX_PARTICLES):
    """Parse raw CSV into structured arrays."""
    raw = df.values
    n_events = len(df)
    
    event_features = raw[:, 1:8].astype(np.float32)        # Columns 1-7
    particle_features = np.zeros(
        (n_events, max_particles, N_FEAT_PER_PARTICLE), dtype=np.float32
    )
    particle_mask = np.zeros((n_events, max_particles), dtype=np.float32)
    
    for i in range(n_events):
        n_obj = int(raw[i, 0])
        n_obj = min(n_obj, max_particles)
        for j in range(n_obj):
            col_start = OBJ_START + j * N_FEAT_PER_PARTICLE
            col_end = col_start + N_FEAT_PER_PARTICLE
            if col_end <= raw.shape[1]:
                particle_features[i, j, :] = raw[i, col_start:col_end]
                particle_mask[i, j] = 1.0
    
    return event_features, particle_features, particle_mask


# Parse
sig_evt, sig_parts, sig_mask = parse_data(sig_df)
bkg_evt, bkg_parts, bkg_mask = parse_data(bkg_df)

# Combine
event_features = np.concatenate([sig_evt, bkg_evt], axis=0)
particle_features = np.concatenate([sig_parts, bkg_parts], axis=0)
particle_mask = np.concatenate([sig_mask, bkg_mask], axis=0)
labels = np.concatenate([
    np.ones(len(sig_evt)), np.zeros(len(bkg_evt))
]).astype(np.float32)

print(f"Total events: {len(labels)}")
print(f"Event features: {event_features.shape}")           # (N, 7)
print(f"Particle features: {particle_features.shape}")     # (N, 50, 4)

# Train/val/test split
(evt_train, evt_test,
 parts_train, parts_test,
 mask_train, mask_test,
 y_train, y_test) = train_test_split(
    event_features, particle_features, particle_mask, labels,
    test_size=0.2, random_state=42, stratify=labels
)

(evt_tr, evt_val,
 parts_tr, parts_val,
 mask_tr, mask_val,
 y_tr, y_val) = train_test_split(
    evt_train, parts_train, mask_train, y_train,
    test_size=0.15, random_state=42
)

print(f"Train: {len(y_tr)}, Val: {len(y_val)}, Test: {len(y_test)}")

# ============================================================
# 2. DATASET AND DATALOADER
# ============================================================

class EventDataset(Dataset):
    """Custom dataset for event classification."""
    def __init__(self, evt_feats, part_feats, mask, labels):
        self.evt = torch.FloatTensor(evt_feats)
        self.parts = torch.FloatTensor(part_feats)
        self.mask = torch.FloatTensor(mask)
        self.labels = torch.FloatTensor(labels).unsqueeze(1)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.evt[idx], self.parts[idx], self.mask[idx], self.labels[idx]


BATCH_SIZE = 64
train_dataset = EventDataset(evt_tr, parts_tr, mask_tr, y_tr)
val_dataset = EventDataset(evt_val, parts_val, mask_val, y_val)
test_dataset = EventDataset(evt_test, parts_test, mask_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ============================================================
# 3. COMPUTE PAIRWISE EDGE FEATURES
# ============================================================
# In a fully connected graph, we precompute edge features for all (i, j) pairs.
# These capture the geometric/kinematic relationship between particles.


def compute_edge_features(particle_feats, mask):
    """
    Compute pairwise edge features for all particle pairs in the event.
    
    For particles i and j with features (pT, eta, phi, flag):
      - Δη_ij = η_i - η_j
      - Δφ_ij = φ_i - φ_j (wrapped to [-π, π])
      - ΔR_ij = sqrt(Δη² + Δφ²)
      - min(pT_i, pT_j) / max(pT_i, pT_j) — pT ratio
      - kT_ij = min(pT_i, pT_j) × ΔR_ij
      - z_ij = min(pT_i, pT_j) / (pT_i + pT_j) — momentum fraction
      - log(ΔR_ij + ε)
      - log(kT_ij + ε)
    
    Parameters:
    -----------
    particle_feats : (batch, N, 4)
    mask : (batch, N)
    
    Returns:
    --------
    edge_features : (batch, N, N, n_edge_features)
    edge_mask : (batch, N, N) — 1 for valid edges, 0 for padding
    """
    pt = particle_feats[:, :, 0]                           # (batch, N)
    eta = particle_feats[:, :, 1]                          # (batch, N)
    phi = particle_feats[:, :, 2]                          # (batch, N)
    flag = particle_feats[:, :, 3]                         # (batch, N)
    
    batch_size, n_parts = pt.shape
    
    # Pairwise differences
    # Δη: (batch, N, N)
    d_eta = eta.unsqueeze(2) - eta.unsqueeze(1)            # η_i - η_j
    
    # Δφ (wrapped to [-π, π]): (batch, N, N)
    d_phi_raw = phi.unsqueeze(2) - phi.unsqueeze(1)        # φ_i - φ_j
    d_phi = torch.atan2(torch.sin(d_phi_raw), torch.cos(d_phi_raw))  # Wrap
    
    # ΔR: (batch, N, N)
    d_R = torch.sqrt(d_eta**2 + d_phi**2 + 1e-8)
    
    # pT values for pairs
    pt_i = pt.unsqueeze(2).expand(-1, -1, n_parts)        # (batch, N, N)
    pt_j = pt.unsqueeze(1).expand(-1, n_parts, -1)        # (batch, N, N)
    
    # min/max pT
    pt_min = torch.min(pt_i, pt_j)
    pt_max = torch.max(pt_i, pt_j).clamp(min=1e-8)
    pt_sum = (pt_i + pt_j).clamp(min=1e-8)
    
    # Derived quantities
    pt_ratio = pt_min / pt_max                             # pT ratio [0, 1]
    kT = pt_min * d_R                                      # kT measure
    z = pt_min / pt_sum                                    # Momentum sharing [0, 0.5]
    log_dR = torch.log(d_R + 1e-5)                         # Log ΔR
    log_kT = torch.log(kT + 1e-5)                          # Log kT
    
    # Flag differences (categorical interaction)
    flag_i = flag.unsqueeze(2).expand(-1, -1, n_parts)
    flag_j = flag.unsqueeze(1).expand(-1, n_parts, -1)
    flag_same = (flag_i == flag_j).float()                 # 1 if same type
    
    # Stack all edge features: (batch, N, N, n_edge_feat)
    edge_features = torch.stack([
        d_eta,            # 0: Δη
        d_phi,            # 1: Δφ
        d_R,              # 2: ΔR
        pt_ratio,         # 3: pT ratio
        kT,               # 4: kT
        z,                # 5: momentum fraction
        log_dR,           # 6: log(ΔR)
        log_kT,           # 7: log(kT)
        flag_same,        # 8: same particle type indicator
    ], dim=-1)            # (batch, N, N, 9)
    
    # Edge mask: both particles must be real (not padding)
    edge_mask = mask.unsqueeze(1) * mask.unsqueeze(2)      # (batch, N, N)
    
    # Remove self-loops (diagonal = 0)
    eye = torch.eye(n_parts).unsqueeze(0)                  # (1, N, N)
    edge_mask = edge_mask * (1 - eye)                      # Zero out diagonal
    
    return edge_features, edge_mask


# Number of edge features we compute
N_EDGE_FEATURES = 9
print(f"Edge features per pair: {N_EDGE_FEATURES}")
print(f"Edge feature names: [Δη, Δφ, ΔR, pT_ratio, kT, z, log_ΔR, log_kT, same_flag]")

# ============================================================
# 4. FULLY CONNECTED GRAPH NETWORK — MODEL DEFINITION
# ============================================================


class EdgeNetwork(nn.Module):
    """
    Edge network: computes messages along edges.
    
    Input: concatenation of [node_i, node_j, edge_features]
    Output: message vector for edge (i, j)
    """
    def __init__(self, node_dim, edge_feat_dim, message_dim, hidden_dim=128):
        """
        Parameters:
        -----------
        node_dim : int — dimension of node features
        edge_feat_dim : int — dimension of raw edge features (9)
        message_dim : int — dimension of output message
        hidden_dim : int — hidden layer size
        """
        super(EdgeNetwork, self).__init__()
        
        # Input: [h_i, h_j, e_ij] → message
        input_dim = 2 * node_dim + edge_feat_dim
        
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),              # Project to hidden
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),             # Second hidden layer
            nn.ReLU(),
            nn.Linear(hidden_dim, message_dim)             # Output message
        )
    
    def forward(self, h_i, h_j, edge_feats):
        """
        h_i : (batch, N, N, node_dim) — sender node features (expanded)
        h_j : (batch, N, N, node_dim) — receiver node features (expanded)
        edge_feats : (batch, N, N, edge_feat_dim)
        
        Returns: (batch, N, N, message_dim) — messages for all edges
        """
        # Concatenate all inputs along feature dimension
        inputs = torch.cat([h_i, h_j, edge_feats], dim=-1)
        return self.mlp(inputs)


class NodeNetwork(nn.Module):
    """
    Node network: updates node features using aggregated messages.
    
    Input: [current_node_features, aggregated_messages]
    Output: updated node features
    """
    def __init__(self, node_dim, message_dim, hidden_dim=128):
        super(NodeNetwork, self).__init__()
        
        # Input: [h_i, aggregated_msg_i]
        self.mlp = nn.Sequential(
            nn.Linear(node_dim + message_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim)                # Output same dim as input (residual)
        )
        
        self.layer_norm = nn.LayerNorm(node_dim)
    
    def forward(self, h, agg_messages):
        """
        h : (batch, N, node_dim) — current node features
        agg_messages : (batch, N, message_dim) — aggregated incoming messages
        
        Returns: (batch, N, node_dim) — updated node features
        """
        inputs = torch.cat([h, agg_messages], dim=-1)
        update = self.mlp(inputs)
        h_new = h + update                                 # Residual connection
        h_new = self.layer_norm(h_new)                     # Normalize
        return h_new


class FullyConnectedGNNLayer(nn.Module):
    """
    One layer of the fully connected graph neural network.
    
    Steps:
      1. Compute messages on ALL edges using EdgeNetwork
      2. Aggregate messages at each node (mean over senders)
      3. Update node features using NodeNetwork
    
    Optionally also updates edge features (edge_update=True).
    """
    def __init__(self, node_dim, edge_feat_dim, message_dim=64,
                 hidden_dim=128, edge_update=True):
        """
        Parameters:
        -----------
        node_dim : int — node feature dimension
        edge_feat_dim : int — edge feature dimension
        message_dim : int — message vector dimension
        hidden_dim : int — hidden layer width
        edge_update : bool — whether to update edge features too
        """
        super(FullyConnectedGNNLayer, self).__init__()
        
        self.edge_update = edge_update
        
        # Edge network: computes messages
        self.edge_net = EdgeNetwork(node_dim, edge_feat_dim, message_dim, hidden_dim)
        
        # Node network: updates nodes with aggregated messages
        self.node_net = NodeNetwork(node_dim, message_dim, hidden_dim)
        
        # Optional: edge update network
        if edge_update:
            self.edge_update_net = nn.Sequential(
                nn.Linear(message_dim + edge_feat_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, edge_feat_dim)       # Keep same edge dim
            )
            self.edge_norm = nn.LayerNorm(edge_feat_dim)
    
    def forward(self, h, edge_feats, edge_mask):
        """
        Parameters:
        -----------
        h : (batch, N, node_dim) — node features
        edge_feats : (batch, N, N, edge_feat_dim) — edge features
        edge_mask : (batch, N, N) — valid edges (1 = real, 0 = padded)
        
        Returns:
        --------
        h_new : (batch, N, node_dim) — updated node features
        edge_feats_new : (batch, N, N, edge_feat_dim) — updated edge features
        """
        batch_size, n_nodes, node_dim = h.shape
        
        # --- Step 1: Compute messages on all edges ---
        # Expand node features for all pairs: (batch, N, N, node_dim)
        h_i = h.unsqueeze(2).expand(-1, -1, n_nodes, -1)  # Sender (row i)
        h_j = h.unsqueeze(1).expand(-1, n_nodes, -1, -1)  # Receiver (col j)
        
        # Compute messages: (batch, N, N, message_dim)
        messages = self.edge_net(h_i, h_j, edge_feats)
        
        # Zero out messages for invalid edges (padding)
        messages = messages * edge_mask.unsqueeze(-1)      # (batch, N, N, msg_dim)
        
        # --- Step 2: Aggregate messages at each node ---
        # Sum messages coming INTO each node j: sum over senders i
        # messages[:, i, j, :] = message from i to j
        # We want: for each j, sum over all i
        msg_sum = messages.sum(dim=1)                      # (batch, N, message_dim)
        
        # Normalize by number of real senders
        n_senders = edge_mask.sum(dim=1, keepdim=False).clamp(min=1)  # (batch, N)
        msg_agg = msg_sum / n_senders.unsqueeze(-1)        # (batch, N, message_dim)
        
        # --- Step 3: Update node features ---
        h_new = self.node_net(h, msg_agg)
        
        # --- Step 4 (optional): Update edge features ---
        if self.edge_update:
            # Use messages to update edges
            edge_input = torch.cat([messages, edge_feats], dim=-1)
            edge_update = self.edge_update_net(edge_input)
            edge_feats_new = edge_feats + edge_update      # Residual
            edge_feats_new = self.edge_norm(edge_feats_new)
            edge_feats_new = edge_feats_new * edge_mask.unsqueeze(-1)
        else:
            edge_feats_new = edge_feats
        
        return h_new, edge_feats_new


class FullyConnectedGNN(nn.Module):
    """
    Fully Connected Graph Neural Network for event classification.
    
    Architecture:
      1. Node encoder: particle features (4) → node_dim
      2. Edge encoder: pairwise features (9) → edge_dim
      3. N message-passing layers on complete graph
      4. Global pooling: attention-weighted sum over nodes
      5. Concatenate with event-level features
      6. Classifier MLP → binary output
    """
    def __init__(self, particle_dims=4, event_dims=7, n_edge_feats=9,
                 node_dim=64, edge_dim=32, message_dim=64,
                 n_layers=4, hidden_dim=128, dropout=0.1):
        """
        Parameters:
        -----------
        particle_dims : int — raw features per particle (4)
        event_dims : int — event-level features (7)
        n_edge_feats : int — raw pairwise features (9)
        node_dim : int — latent node representation size
        edge_dim : int — latent edge representation size
        message_dim : int — size of messages passed between nodes
        n_layers : int — number of message-passing layers
        hidden_dim : int — hidden layer size in MLPs
        dropout : float — dropout rate
        """
        super(FullyConnectedGNN, self).__init__()
        
        self.n_layers = n_layers
        self.node_dim = node_dim
        self.edge_dim = edge_dim
        
        # --- Node encoder ---
        # Maps raw particle features to node embedding
        self.node_encoder = nn.Sequential(
            nn.Linear(particle_dims, hidden_dim),          # (4) → (hidden)
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim),               # (hidden) → (node_dim)
            nn.LayerNorm(node_dim)
        )
        
        # --- Edge encoder ---
        # Maps raw pairwise features to edge embedding
        self.edge_encoder = nn.Sequential(
            nn.Linear(n_edge_feats, hidden_dim),           # (9) → (hidden)
            nn.ReLU(),
            nn.Linear(hidden_dim, edge_dim),               # (hidden) → (edge_dim)
            nn.LayerNorm(edge_dim)
        )
        
        # --- Message-passing layers ---
        self.gnn_layers = nn.ModuleList([
            FullyConnectedGNNLayer(
                node_dim=node_dim,
                edge_feat_dim=edge_dim,
                message_dim=message_dim,
                hidden_dim=hidden_dim,
                edge_update=(i < n_layers - 1)             # Don't update edges in last layer
            )
            for i in range(n_layers)
        ])
        
        # --- Global pooling (attention-based) ---
        # Learn which nodes are most important for classification
        self.attention_pool = nn.Sequential(
            nn.Linear(node_dim, node_dim // 2),
            nn.Tanh(),
            nn.Linear(node_dim // 2, 1)                    # Scalar attention score
        )
        
        # --- Event feature encoder ---
        self.event_encoder = nn.Sequential(
            nn.Linear(event_dims, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        
        # --- Classifier head ---
        clf_input_dim = node_dim + 32                      # Pooled nodes + event features
        self.classifier = nn.Sequential(
            nn.Linear(clf_input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()                                   # Binary probability
        )
    
    def forward(self, event_feats, particle_feats, mask):
        """
        Parameters:
        -----------
        event_feats : (batch, 7) — event-level features
        particle_feats : (batch, N, 4) — per-particle (pT, eta, phi, flag)
        mask : (batch, N) — particle mask (1=real, 0=pad)
        
        Returns:
        --------
        (batch, 1) — signal probability
        """
        batch_size, n_nodes, _ = particle_feats.shape
        
        # --- Compute edge features ---
        # (batch, N, N, 9) and (batch, N, N)
        raw_edge_feats, edge_mask = compute_edge_features(particle_feats, mask)
        
        # --- Encode nodes ---
        # (batch, N, 4) → (batch, N, node_dim)
        h = self.node_encoder(particle_feats)
        h = h * mask.unsqueeze(-1)                         # Zero out padded nodes
        
        # --- Encode edges ---
        # (batch, N, N, 9) → (batch, N, N, edge_dim)
        edge_feats = self.edge_encoder(raw_edge_feats)
        edge_feats = edge_feats * edge_mask.unsqueeze(-1)  # Zero out invalid edges
        
        # --- Message passing ---
        for layer in self.gnn_layers:
            h, edge_feats = layer(h, edge_feats, edge_mask)
            h = h * mask.unsqueeze(-1)                     # Maintain padding zeros
        
        # --- Global pooling (attention-weighted sum) ---
        # Compute attention scores per node
        attn_scores = self.attention_pool(h)               # (batch, N, 1)
        
        # Mask padded nodes (set score to -inf before softmax)
        attn_scores = attn_scores + (1 - mask.unsqueeze(-1)) * (-1e9)
        
        # Softmax over nodes
        attn_weights = torch.softmax(attn_scores, dim=1)   # (batch, N, 1)
        attn_weights = attn_weights * mask.unsqueeze(-1)   # Zero padded (safety)
        
        # Weighted sum of node features
        graph_repr = (attn_weights * h).sum(dim=1)         # (batch, node_dim)
        
        # --- Encode event features ---
        evt_repr = self.event_encoder(event_feats)         # (batch, 32)
        
        # --- Classify ---
        combined = torch.cat([graph_repr, evt_repr], dim=1)  # (batch, node_dim + 32)
        output = self.classifier(combined)                 # (batch, 1)
        
        return output


# ============================================================
# 5. INSTANTIATE MODEL
# ============================================================

print("=" * 60)
print("FULLY CONNECTED GRAPH NEURAL NETWORK")
print("=" * 60)

model = FullyConnectedGNN(
    particle_dims=N_FEAT_PER_PARTICLE,     # 4: pT, eta, phi, flag
    event_dims=7,                          # 7 event-level features
    n_edge_feats=N_EDGE_FEATURES,          # 9 pairwise features
    node_dim=64,                           # Node embedding dimension
    edge_dim=32,                           # Edge embedding dimension
    message_dim=64,                        # Message vector size
    n_layers=4,                            # 4 message-passing rounds
    hidden_dim=128,                        # Hidden MLP width
    dropout=0.15                           # Dropout rate
).to(DEVICE)

# Model statistics
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n  Node dimension: 64")
print(f"  Edge dimension: 32")
print(f"  Message dimension: 64")
print(f"  Message-passing layers: 4")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Quick shape check with a dummy batch
print("\n  Shape check:")
dummy_evt = torch.randn(2, 7)
dummy_parts = torch.randn(2, MAX_PARTICLES, 4)
dummy_mask = torch.ones(2, MAX_PARTICLES)
dummy_mask[:, 30:] = 0                                     # Mask last 20 as padding
with torch.no_grad():
    dummy_out = model(dummy_evt, dummy_parts, dummy_mask)
print(f"    Input particles: {dummy_parts.shape}")
print(f"    Output: {dummy_out.shape}")
print(f"    Output values: {dummy_out.squeeze().numpy()}")

# ============================================================
# 6. TRAINING LOOP
# ============================================================

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

criterion = nn.BCELoss()                                   # Binary cross-entropy
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

EPOCHS = 60
PATIENCE = 12
best_val_loss = float('inf')
patience_counter = 0
best_state = None

history = {
    'train_loss': [], 'val_loss': [], 'val_auc': [],
    'train_acc': [], 'val_acc': []
}

for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    train_loss = 0.0
    train_correct = 0
    n_train = 0
    
    for evt, parts, mask, lbl in train_loader:
        optimizer.zero_grad()                              # Clear gradients
        output = model(evt, parts, mask)                   # Forward pass
        loss = criterion(output, lbl)                      # Compute loss
        loss.backward()                                    # Backpropagation
        
        # Gradient clipping — important for GNNs to prevent instability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()                                   # Update weights
        
        train_loss += loss.item() * evt.size(0)
        train_correct += ((output >= 0.5).float() == lbl).sum().item()
        n_train += evt.size(0)
    
    train_loss /= n_train
    train_acc = train_correct / n_train
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    
    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_labels_list = []
    n_val = 0
    
    with torch.no_grad():
        for evt, parts, mask, lbl in val_loader:
            output = model(evt, parts, mask)
            loss = criterion(output, lbl)
            val_loss += loss.item() * evt.size(0)
            val_preds.append(output.numpy())
            val_labels_list.append(lbl.numpy())
            n_val += evt.size(0)
    
    val_loss /= n_val
    val_preds_arr = np.concatenate(val_preds).ravel()
    val_labels_arr = np.concatenate(val_labels_list).ravel()
    val_auc = roc_auc_score(val_labels_arr, val_preds_arr)
    val_acc = accuracy_score(val_labels_arr, (val_preds_arr >= 0.5).astype(int))
    
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['val_acc'].append(val_acc)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break
    
    if (epoch + 1) % 5 == 0:
        lr_now = optimizer.param_groups[0]['lr']
        print(f"  Epoch {epoch+1:3d}/{EPOCHS}: "
              f"train_loss={train_loss:.4f}, "
              f"val_loss={val_loss:.4f}, "
              f"val_auc={val_auc:.4f}, "
              f"val_acc={val_acc:.4f}, "
              f"lr={lr_now:.6f}")

# Restore best model
model.load_state_dict(best_state)
print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

# ============================================================
# 7. EVALUATION ON TEST SET
# ============================================================

print("\n" + "=" * 60)
print("EVALUATION ON TEST SET")
print("=" * 60)

model.eval()
all_preds = []
with torch.no_grad():
    for evt, parts, mask, lbl in test_loader:
        output = model(evt, parts, mask)
        all_preds.append(output.numpy())

y_prob = np.concatenate(all_preds).ravel()
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"\nFully Connected GNN Results:")
print(f"  Accuracy: {acc:.4f}")
print(f"  AUC-ROC:  {auc:.4f}")

# Working points
print(f"\nWorking points:")
print(f"  {'Threshold':<12} {'Sig Eff':<12} {'Bkg Rej':<12}")
print(f"  {'-'*36}")
for threshold in [0.1, 0.3, 0.5, 0.7, 0.9, 0.95]:
    preds_wp = (y_prob >= threshold).astype(int)
    sig_eff = preds_wp[y_test == 1].mean()
    bkg_rej = 1.0 - preds_wp[y_test == 0].mean()
    print(f"  {threshold:<12.2f} {sig_eff:<12.4f} {bkg_rej:<12.4f}")

# ============================================================
# 8. ROC CURVE
# ============================================================

fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Standard ROC
ax1.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'FC-GNN (AUC = {auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
ax1.set_xlabel('False Positive Rate (Background Efficiency)')
ax1.set_ylabel('True Positive Rate (Signal Efficiency)')
ax1.set_title('ROC Curve — Fully Connected GNN')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Background rejection vs signal efficiency
bkg_rej = 1.0 / (fpr + 1e-10)
valid = fpr > 0
ax2.plot(tpr[valid], bkg_rej[valid], color='darkorange', lw=2)
ax2.set_xlabel('Signal Efficiency')
ax2.set_ylabel('Background Rejection (1/FPR)')
ax2.set_yscale('log')
ax2.set_xlim([0, 1])
ax2.set_ylim([1, 1e4])
ax2.set_title('Signal Eff. vs Background Rejection')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('roc_fc_gnn.pdf')
plt.show()

# ============================================================
# 9. TRAINING CURVES
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train', color='blue')
axes[0].plot(history['val_loss'], label='Val', color='orange')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', color='blue')
axes[1].plot(history['val_acc'], label='Val', color='orange')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# AUC
axes[2].plot(history['val_auc'], label='Val AUC', color='green')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('AUC')
axes[2].set_title('Validation AUC'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fc_gnn_training_curves.pdf')
plt.show()

# ============================================================
# 10. OUTPUT SCORE DISTRIBUTION
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))
sig_scores = y_prob[y_test == 1]
bkg_scores = y_prob[y_test == 0]

ax.hist(sig_scores, bins=50, alpha=0.6, density=True, label='Signal', color='blue')
ax.hist(bkg_scores, bins=50, alpha=0.6, density=True, label='Background', color='red')
ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Threshold = 0.5')
ax.set_xlabel('GNN Output Score')
ax.set_ylabel('Normalized Distribution')
ax.set_title('FC-GNN — Discriminant Output')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fc_gnn_output_distribution.pdf')
plt.show()

# ============================================================
# 11. ATTENTION WEIGHT ANALYSIS
# ============================================================
# Examine which particles the model pays most attention to during pooling

print("\n" + "=" * 60)
print("ATTENTION WEIGHT ANALYSIS")
print("=" * 60)


def extract_attention_weights(model, evt, parts, mask):
    """Extract attention pooling weights for a batch."""
    model.eval()
    with torch.no_grad():
        batch_size, n_nodes, _ = parts.shape
        
        # Compute edge features
        raw_edge_feats, edge_mask = compute_edge_features(parts, mask)
        
        # Encode
        h = model.node_encoder(parts) * mask.unsqueeze(-1)
        edge_feats = model.edge_encoder(raw_edge_feats) * edge_mask.unsqueeze(-1)
        
        # Message passing
        for layer in model.gnn_layers:
            h, edge_feats = layer(h, edge_feats, edge_mask)
            h = h * mask.unsqueeze(-1)
        
        # Attention scores
        attn_scores = model.attention_pool(h)
        attn_scores = attn_scores + (1 - mask.unsqueeze(-1)) * (-1e9)
        attn_weights = torch.softmax(attn_scores, dim=1)
        attn_weights = attn_weights * mask.unsqueeze(-1)
    
    return attn_weights.squeeze(-1).numpy()                # (batch, N)


# Get attention weights for test signal and background
n_analyze = min(300, len(test_dataset))
analyze_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

all_attns = []
all_pts = []
all_true_labels = []

for evt, parts, mask, lbl in analyze_loader:
    attns = extract_attention_weights(model, evt, parts, mask)
    n_real = mask.numpy()                                  # (batch, N)
    
    for i in range(evt.size(0)):
        n_part = int(mask[i].sum())
        all_attns.extend(attns[i, :n_part].tolist())
        all_pts.extend(parts[i, :n_part, 0].numpy().tolist())
        all_true_labels.extend([lbl[i].item()] * n_part)
    
    if len(all_attns) > 10000:
        break

all_attns = np.array(all_attns)
all_pts = np.array(all_pts)
all_true_labels = np.array(all_true_labels)

# Plot: attention weight vs pT, colored by class
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# Scatter plot
sig_mask_attn = all_true_labels == 1
bkg_mask_attn = all_true_labels == 0

ax1.scatter(all_pts[sig_mask_attn][:2000], all_attns[sig_mask_attn][:2000],
            alpha=0.2, s=10, c='blue', label='Signal particles')
ax1.scatter(all_pts[bkg_mask_attn][:2000], all_attns[bkg_mask_attn][:2000],
            alpha=0.2, s=10, c='red', label='Background particles')
ax1.set_xlabel('Particle pT')
ax1.set_ylabel('Attention Weight')
ax1.set_title('Attention Weight vs pT')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Histogram of attention weights
ax2.hist(all_attns[sig_mask_attn], bins=50, alpha=0.5, density=True,
         label='Signal', color='blue')
ax2.hist(all_attns[bkg_mask_attn], bins=50, alpha=0.5, density=True,
         label='Background', color='red')
ax2.set_xlabel('Attention Weight')
ax2.set_ylabel('Density')
ax2.set_title('Attention Weight Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Mean attention vs particle rank (pT-ordered)
# Shows if model focuses on leading or subleading particles
max_rank = 20
sig_attn_by_rank = np.zeros(max_rank)
bkg_attn_by_rank = np.zeros(max_rank)
sig_count_by_rank = np.zeros(max_rank)
bkg_count_by_rank = np.zeros(max_rank)

for evt, parts, mask, lbl in DataLoader(test_dataset, batch_size=64, shuffle=False):
    attns = extract_attention_weights(model, evt, parts, mask)
    for i in range(evt.size(0)):
        n_part = int(mask[i].sum())
        for rank in range(min(n_part, max_rank)):
            if lbl[i].item() == 1:
                sig_attn_by_rank[rank] += attns[i, rank]
                sig_count_by_rank[rank] += 1
            else:
                bkg_attn_by_rank[rank] += attns[i, rank]
                bkg_count_by_rank[rank] += 1

sig_attn_by_rank /= np.clip(sig_count_by_rank, 1, None)
bkg_attn_by_rank /= np.clip(bkg_count_by_rank, 1, None)

ax3.plot(range(1, max_rank+1), sig_attn_by_rank, 'bo-', lw=2, markersize=6, label='Signal')
ax3.plot(range(1, max_rank+1), bkg_attn_by_rank, 'rs-', lw=2, markersize=6, label='Background')
ax3.set_xlabel('Particle Rank (pT-ordered, 1=leading)')
ax3.set_ylabel('Mean Attention Weight')
ax3.set_title('Attention vs Particle Rank')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fc_gnn_attention_analysis.pdf')
plt.show()

# ============================================================
# 12. EDGE IMPORTANCE ANALYSIS
# ============================================================
# Visualize which particle pairs have the strongest learned interactions

print("\n" + "=" * 60)
print("EDGE IMPORTANCE ANALYSIS")
print("=" * 60)


def get_edge_messages_norm(model, evt, parts, mask):
    """
    Get L2-norm of messages on each edge (after last GNN layer).
    Larger norm = more important interaction.
    """
    model.eval()
    with torch.no_grad():
        batch_size, n_nodes, _ = parts.shape
        
        raw_edge_feats, edge_mask = compute_edge_features(parts, mask)
        h = model.node_encoder(parts) * mask.unsqueeze(-1)
        edge_feats = model.edge_encoder(raw_edge_feats) * edge_mask.unsqueeze(-1)
        
        # Run through all but last layer normally
        for layer in model.gnn_layers[:-1]:
            h, edge_feats = layer(h, edge_feats, edge_mask)
            h = h * mask.unsqueeze(-1)
        
        # Last layer: extract messages before aggregation
        last_layer = model.gnn_layers[-1]
        h_i = h.unsqueeze(2).expand(-1, -1, n_nodes, -1)
        h_j = h.unsqueeze(1).expand(-1, n_nodes, -1, -1)
        messages = last_layer.edge_net(h_i, h_j, edge_feats)
        messages = messages * edge_mask.unsqueeze(-1)
        
        # Norm of each message: (batch, N, N)
        msg_norm = messages.norm(dim=-1)
    
    return msg_norm.numpy(), edge_mask.numpy()


# Analyze a few events
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for col in range(3):
    for row, (label_val, title) in enumerate([(1, 'Signal'), (0, 'Background')]):
        idx = np.where(y_test == label_val)[0][col]
        
        evt_t = torch.FloatTensor(parts_test[idx:idx+1])
        mask_t = torch.FloatTensor(mask_test[idx:idx+1])
        evt_feat_t = torch.FloatTensor(evt_test[idx:idx+1])
        
        msg_norm, e_mask = get_edge_messages_norm(model, evt_feat_t, evt_t, mask_t)
        msg_norm = msg_norm[0]                             # (N, N)
        e_mask = e_mask[0]
        
        n_real = int(mask_test[idx].sum())
        msg_sub = msg_norm[:n_real, :n_real]               # Only real particles
        
        ax = axes[row, col]
        im = ax.imshow(msg_sub, cmap='viridis', aspect='equal')
        ax.set_xlabel('Particle j')
        ax.set_ylabel('Particle i')
        ax.set_title(f'{title} event #{idx}\n(score={y_prob[idx]:.3f})')
        plt.colorbar(im, ax=ax, fraction=0.046, label='|Message|')

plt.suptitle('Edge Message Magnitudes (Fully Connected Graph)\n'
             'Brighter = stronger particle interaction', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fc_gnn_edge_importance.pdf', bbox_inches='tight')
plt.show()

# ============================================================
# 13. AVERAGE MESSAGE STRENGTH vs ΔR
# ============================================================
# Do nearby or far-apart particles interact more strongly?

print("\nComputing message strength vs ΔR...")

all_dR = []
all_msg_norms = []

for evt, parts, mask, lbl in DataLoader(test_dataset, batch_size=32, shuffle=False):
    msg_norm, e_mask = get_edge_messages_norm(model, evt, parts, mask)
    
    # Compute ΔR for all pairs
    eta = parts[:, :, 1].numpy()
    phi = parts[:, :, 2].numpy()
    
    for i in range(evt.size(0)):
        n_real = int(mask[i].sum())
        for a in range(n_real):
            for b in range(a+1, n_real):
                deta = eta[i, a] - eta[i, b]
                dphi = phi[i, a] - phi[i, b]
                dphi = np.arctan2(np.sin(dphi), np.cos(dphi))
                dR = np.sqrt(deta**2 + dphi**2)
                mn = (msg_norm[i, a, b] + msg_norm[i, b, a]) / 2  # Symmetrize
                all_dR.append(dR)
                all_msg_norms.append(mn)
    
    if len(all_dR) > 50000:
        break

all_dR = np.array(all_dR)
all_msg_norms = np.array(all_msg_norms)

# Bin by ΔR
dR_bins = np.linspace(0, 2.0, 20)
dR_centers = 0.5 * (dR_bins[:-1] + dR_bins[1:])
mean_msg = np.zeros(len(dR_centers))

for b in range(len(dR_centers)):
    bin_mask = (all_dR >= dR_bins[b]) & (all_dR < dR_bins[b+1])
    if bin_mask.sum() > 0:
        mean_msg[b] = all_msg_norms[bin_mask].mean()

plt.figure(figsize=(8, 5))
plt.plot(dR_centers, mean_msg, 'o-', color='darkorange', lw=2, markersize=7)
plt.xlabel(r'$\Delta R$ between particle pair')
plt.ylabel('Mean |Message| (interaction strength)')
plt.title('Learned Interaction Strength vs Particle Separation')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fc_gnn_interaction_vs_dR.pdf')
plt.show()

print("  → Shows whether the GNN learns to focus on nearby or distant particles")

# ============================================================
# 14. SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("SUMMARY — FULLY CONNECTED GRAPH NEURAL NETWORK")
print("=" * 60)
print(f"""
Architecture:
  - Node encoder:    particle features (4) → node_dim (64)
  - Edge encoder:    pairwise features (9) → edge_dim (32)
  - GNN layers:      4 × FullyConnectedGNNLayer
  - Message dim:     64
  - Pooling:         Attention-weighted sum over nodes
  - Classifier:      MLP [128 → 64 → 32 → 1]

Graph structure:
  - Fully connected (complete graph): every particle connected to all others
  - Self-loops excluded
  - Padded particles masked out

Edge features (9 per pair):
  [Δη, Δφ, ΔR, pT_ratio, kT, z, log(ΔR), log(kT), same_flag]

Results:
  - Accuracy: {acc:.4f}
  - AUC-ROC:  {auc:.4f}
  - Parameters: {total_params:,}

Key observations:
  - Attention weights reveal which particles are most discriminating
  - Edge message norms show which particle interactions the model relies on
  - The model learns the relative importance of nearby vs distant particles
""")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/envs/work_py_ml/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/envs/work_py_ml/lib/python3.11/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/opt/anaconda3/envs/work_py_ml/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 807, in start
    self.io_loop.sta

FileNotFoundError: [Errno 2] No such file or directory: 'CoLBTHydro.csv'